In [ ]:
%run NB00_setup

In [ ]:
%run NB00b_gold_config

# NB04 – KPIs + Reconciliation + Time travel
1) Checks that the latest quality run passed (otherwise it stops). 2) Builds 3 KPI tables. 3) Recomputes the same KPIs from silver and compares. 4) Time travel.

In [ ]:
# ---------- gate: if the latest quality run has a FAIL, KPIs are not built ----------
qr = spark.table("gold_quality_results").where("run_type = 'real'")
last = qr.orderBy(F.desc("run_ts")).first()
if last is None:
    raise Exception("No quality run found. Run NB03 first.")
fails = qr.where(f"run_id = '{last['run_id']}' AND status = 'FAIL'").count()
if fails:
    raise Exception(f"Latest quality run {last['run_id']} has {fails} FAILED checks. KPIs not built.")
print("quality gate OK for run", last["run_id"])

In [ ]:
# ======================= build the KPI tables =======================
t0 = time.time()

def save_overwrite(df, table):
    (df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(table))
    print(f"{table}: {spark.table(table).count()} rows")

ft = spark.table("gold_fact_telemetry")
fe = spark.table("gold_fact_anomaly_episode")
dim_route = spark.table("gold_dim_route").select("route_key", "corridor_id", F.col("name").alias("route_name"))
dim_at    = spark.table("gold_dim_anomaly_type").select("anomaly_type_key", "anomaly_type")
dim_drv   = spark.table("gold_dim_driver").select("driver_key", "driver_id", "full_name",
                                                 "home_depot_id", "risk_level")

# ---- KPI 1: Fuel efficiency per route  (km ÷ litres)  —  grain: route × day ----
fuel_kpi = (ft.groupBy("route_key", "date_key").agg(
                F.sum("distance_km").alias("total_km"),
                F.sum("fuel_used_l").alias("total_litres"),
                F.count("*").alias("readings"))
            .join(dim_route, "route_key", "left")
            .withColumn("km_per_litre", F.when(F.col("total_litres") > 0, F.col("total_km") / F.col("total_litres")))
            .select("route_key", "corridor_id", "route_name", "date_key",
                    "total_km", "total_litres", "km_per_litre", "readings"))
save_overwrite(fuel_kpi, "gold_kpi_fuel_efficiency_route")

# ---- KPI 2: Cold-chain compliance %  -  grain: route x day (+ number of breach episodes) ----
cold_rd = (ft.groupBy("route_key", "date_key").agg(
               F.sum("cold_in_range_cnt").alias("readings_in_range"),
               F.sum("cold_reading_cnt").alias("readings_total")))
cold_ep = (fe.join(dim_at, "anomaly_type_key").where("anomaly_type = 'cold_chain_breach'")
             .groupBy("route_key", F.col("start_date_key").alias("date_key"))
             .agg(F.count("*").alias("breach_episodes")))
cold_kpi = (cold_rd.join(cold_ep, ["route_key", "date_key"], "full_outer")
            .join(dim_route, "route_key", "left")
            .withColumn("readings_in_range", F.coalesce("readings_in_range", F.lit(0)))
            .withColumn("readings_total", F.coalesce("readings_total", F.lit(0)))
            .withColumn("breach_episodes", F.coalesce("breach_episodes", F.lit(0)))
            .withColumn("compliance_pct", F.when(F.col("readings_total") > 0,
                                                 F.col("readings_in_range") * 100.0 / F.col("readings_total")))
            .select("route_key", "corridor_id", "route_name", "date_key",
                    "readings_in_range", "readings_total", "compliance_pct", "breach_episodes"))
save_overwrite(cold_kpi, "gold_kpi_cold_chain")

# ---- KPI 3: Driver safety / idling scorecard  —  grain: driver version × day ----
sc_tel = ft.groupBy("driver_key", "date_key").agg(
    F.sum("idle_minutes").alias("idling_minutes"),
    F.sum("speeding_cnt").alias("speeding_readings"))
sc_ep = (fe.join(dim_at, "anomaly_type_key")
           .groupBy("driver_key", F.col("start_date_key").alias("date_key"))
           .agg(F.sum(F.when(F.col("anomaly_type") == "excessive_idling", 1).otherwise(0)).alias("excessive_idling_episodes"),
                F.sum(F.when(F.col("anomaly_type") == "route_deviation", 1).otherwise(0)).alias("route_deviations")))
score = (sc_tel.join(sc_ep, ["driver_key", "date_key"], "full_outer")
         .join(dim_drv, "driver_key", "left")
         .withColumn("idling_minutes", F.coalesce("idling_minutes", F.lit(0.0)))
         .withColumn("speeding_readings", F.coalesce("speeding_readings", F.lit(0)))
         .withColumn("excessive_idling_episodes", F.coalesce("excessive_idling_episodes", F.lit(0)))
         .withColumn("route_deviations", F.coalesce("route_deviations", F.lit(0)))
         .select("driver_key", "driver_id", "full_name", "home_depot_id", "risk_level", "date_key",
                 "idling_minutes", "excessive_idling_episodes", "speeding_readings", "route_deviations"))
save_overwrite(score, "gold_kpi_driver_scorecard")
log_run("NB04 build_kpis", t0)

print("\nTop 5 routes by fuel efficiency:")
(spark.table("gold_kpi_fuel_efficiency_route").groupBy("corridor_id", "route_name")
    .agg(F.sum("total_km").alias("km"), F.sum("total_litres").alias("litres"))
    .withColumn("km_per_litre", F.col("km") / F.col("litres")).orderBy(F.desc("km_per_litre")).show(5, truncate=False))
print("Overall cold-chain compliance:")
spark.table("gold_kpi_cold_chain").agg(
    (F.sum("readings_in_range") * 100.0 / F.sum("readings_total")).alias("compliance_pct"),
    F.sum("breach_episodes").alias("breach_episodes")).show()

In [ ]:
# ======================= Reconciliation: the same KPIs computed straight from silver (no gold table used) =======================
t0 = time.time()
sv = dedupe_telemetry(spark.table(SRC_TELEMETRY))   # same rule as gold: one event_id = one row
sv = add_corridor(sv).withColumn("date_key", F.date_format("event_time", "yyyyMMdd").cast("int"))

def reconcile(kpi, metric, keys, g, s, gcol, scol, tol=1e-6):
    gg = g.select(*keys, F.col(gcol).alias("g_val"))
    ss = s.select(*keys, F.col(scol).alias("s_val"))
    j = (gg.join(ss, keys, "full_outer")
           .withColumn("g_val", F.coalesce(F.col("g_val").cast("double"), F.lit(0.0)))
           .withColumn("s_val", F.coalesce(F.col("s_val").cast("double"), F.lit(0.0))))
    j = j.withColumn("diff", F.abs(F.col("g_val") - F.col("s_val")))
    j = j.withColumn("bad", F.col("diff") > F.lit(tol) * F.greatest(F.lit(1.0), F.abs(F.col("s_val"))))
    r = j.agg(F.count("*").alias("n"), F.sum(F.col("bad").cast("int")).alias("bad_n"),
              F.sum("g_val").alias("g_total"), F.sum("s_val").alias("s_total"),
              F.max("diff").alias("max_diff")).first()
    bad_n = int(r["bad_n"] or 0)
    return (kpi, metric, int(r["n"]), bad_n, float(r["g_total"] or 0), float(r["s_total"] or 0),
            float(r["max_diff"] or 0), "MATCH" if bad_n == 0 else "MISMATCH")

rows = []
# ---- Fuel efficiency per route ----
g = (spark.table("gold_kpi_fuel_efficiency_route").groupBy("corridor_id")
     .agg(F.sum("total_km").alias("km"), F.sum("total_litres").alias("l"))
     .withColumn("kpl", F.col("km") / F.col("l")))
s = (sv.groupBy("corridor_id").agg(F.sum(distance_expr).alias("km"), F.sum(fuel_expr).alias("l"))
       .withColumn("kpl", F.col("km") / F.col("l")))
for m in ["km", "l", "kpl"]:
    rows.append(reconcile("fuel_efficiency_per_route", {"km": "total_km", "l": "total_litres", "kpl": "km_per_litre"}[m],
                          ["corridor_id"], g, s, m, m))
# ---- Cold-chain compliance ----
g = (spark.table("gold_kpi_cold_chain").groupBy("corridor_id")
     .agg(F.sum("readings_in_range").alias("ok"), F.sum("readings_total").alias("tot"))
     .withColumn("pct", F.when(F.col("tot") > 0, F.col("ok") * 100.0 / F.col("tot"))))
s = (sv.groupBy("corridor_id").agg(F.sum(cold_ok_expr).alias("ok"), F.sum(cold_total_expr).alias("tot"))
       .withColumn("pct", F.when(F.col("tot") > 0, F.col("ok") * 100.0 / F.col("tot"))))
for m, name in [("ok", "readings_in_range"), ("tot", "readings_total"), ("pct", "compliance_pct")]:
    rows.append(reconcile("cold_chain_compliance", name, ["corridor_id"], g, s, m, m))
# overall compliance % (single number)
g_all = spark.table("gold_kpi_cold_chain").agg(F.sum("readings_in_range").alias("ok"), F.sum("readings_total").alias("tot"))
s_all = sv.agg(F.sum(cold_ok_expr).alias("ok"), F.sum(cold_total_expr).alias("tot"))
g_all = g_all.withColumn("k", F.lit(1)).withColumn("pct", F.col("ok") * 100.0 / F.col("tot"))
s_all = s_all.withColumn("k", F.lit(1)).withColumn("pct", F.col("ok") * 100.0 / F.col("tot"))
rows.append(reconcile("cold_chain_compliance", "overall_compliance_pct", ["k"], g_all, s_all, "pct", "pct"))
# ---- Driver scorecard ----
g = (spark.table("gold_kpi_driver_scorecard").groupBy("driver_id", "date_key")
     .agg(F.sum("idling_minutes").alias("idle"), F.sum("excessive_idling_episodes").alias("epi"),
          F.sum("speeding_readings").alias("spd"), F.sum("route_deviations").alias("dev")))
s_tel = (sv.groupBy(F.col("assigned_driver_id").alias("driver_id"), "date_key")
           .agg(F.sum(idle_min_expr).alias("idle"), F.sum(speeding_expr).alias("spd")))
s_ep = (spark.table(SRC_EPISODES)
        .join(spark.table("ref_vehicles").select("vehicle_id", F.col("assigned_driver_id").alias("driver_id")), "vehicle_id")
        .withColumn("date_key", F.date_format("start_time", "yyyyMMdd").cast("int"))
        .groupBy("driver_id", "date_key")
        .agg(F.sum(F.when(F.col("anomaly_type") == "excessive_idling", 1).otherwise(0)).alias("epi"),
             F.sum(F.when(F.col("anomaly_type") == "route_deviation", 1).otherwise(0)).alias("dev")))
s = s_tel.join(s_ep, ["driver_id", "date_key"], "full_outer")
for m, name in [("idle", "idling_minutes"), ("epi", "excessive_idling_episodes"),
                ("spd", "speeding_readings"), ("dev", "route_deviations")]:
    rows.append(reconcile("driver_scorecard", name, ["driver_id", "date_key"], g, s, m, m))

rec = spark.createDataFrame(
    [r + (datetime.datetime.utcnow(),) for r in rows],
    "kpi string, metric string, groups_compared long, groups_mismatched long, gold_total double, "
    "silver_total double, max_abs_diff double, status string, run_ts timestamp")
rec.write.mode("append").format("delta").saveAsTable("gold_reconciliation_results")
rec.drop("run_ts").show(50, truncate=False)
log_run("NB04 reconciliation", t0)
bad_rows = [r for r in rows if r[7] != "MATCH"]
print("ALL KPIs MATCH silver" if not bad_rows else f"{len(bad_rows)} metrics MISMATCH - see the table above")

In [ ]:
# ======================= Time travel =======================
for t in ["gold_dim_driver", "gold_fact_trip", "gold_kpi_driver_scorecard"]:
    print("=" * 10, t)
    show_history(t, 10)

h = spark.sql("DESCRIBE HISTORY gold_dim_driver").orderBy("version").collect()
versions = [r["version"] for r in h]
v_old = versions[1] if len(versions) > 1 else versions[0]
v_new = versions[-1]
print(f"gold_dim_driver: version {v_old} (after the v1 load) vs version {v_new} (current)")
for v in [v_old, v_new]:
    r = spark.sql(f"SELECT COUNT(*) AS total_rows, SUM(CASE WHEN is_current THEN 1 ELSE 0 END) AS current_rows, "
                  f"SUM(CASE WHEN is_current THEN 0 ELSE 1 END) AS closed_rows FROM gold_dim_driver VERSION AS OF {v}")
    print("version", v); r.show()

did = (spark.table("gold_dim_driver").groupBy("driver_id").count().where("count > 1")
       .select("driver_id").limit(1).first()["driver_id"])
print("The same driver", did, "in the old version:")
spark.sql(f"SELECT driver_key, driver_id, home_depot_id, risk_level, effective_from, effective_to, is_current "
          f"FROM gold_dim_driver VERSION AS OF {v_old} WHERE driver_id = '{did}'").show(truncate=False)
print("And in the current version:")
spark.sql(f"SELECT driver_key, driver_id, home_depot_id, risk_level, effective_from, effective_to, is_current "
          f"FROM gold_dim_driver WHERE driver_id = '{did}' ORDER BY effective_from").show(truncate=False)

In [ ]:
# ======================= table summary =======================
tables = ["gold_dim_date", "gold_dim_time", "gold_dim_anomaly_type", "gold_dim_vehicle", "gold_dim_route",
          "gold_dim_driver", "gold_fact_telemetry", "gold_fact_trip", "gold_fact_anomaly_episode",
          "gold_kpi_fuel_efficiency_route", "gold_kpi_cold_chain", "gold_kpi_driver_scorecard",
          "gold_quality_results", "gold_reconciliation_results", "gold_job_run_times"]
for t in tables:
    print(f"{t:36s} rows = {spark.table(t).count()}")